<a href="https://colab.research.google.com/github/guillaumevalette2-hash/mse_gh/blob/main/pgt_ostat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import time
from itertools import product
from sklearn.linear_model import Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import roc_auc_score, accuracy_score
from scipy.optimize import minimize, LinearConstraint

# ══════════════════════════════════════════════════════════════════════════════
# PLONGEMENT SPECTRAL vs DEUX NAPPES  {P_sep=0} ∪ {P_sep=ε}
#
# Question : les deux nappes S'ÉLOIGNENT-elles dans le plongement ?
# Protocole :
#   1. cloud 50/50 sur les deux nappes (P_sep renormalisé std=1, Gauss-Newton) ;
#   2. diagnostic de séparation AVANT (espace ambiant) ;
#   3. classification QP (semi-interpolation Sobolev, expert polynomial) AVANT ;
#   4. plongement spectral (projections aléatoires + base polynomiale + Gram
#      Sobolev, vecteurs propres normalisés) — le programme historique ;
#   5. diagnostic de séparation APRÈS (espace plongé) ;
#   6. classification QP APRÈS (petit degré : si le plongement redresse, un
#      degré bas doit suffire) ;
#   7. baseline Ridge poly avant/après (RBF supprimé).
# ══════════════════════════════════════════════════════════════════════════════

params = {
    # ── GÉOMÉTRIE ─────────────────────────────────────────────────────────────
    "n_ambiant":    4,
    "deg_P":        5,          # degré de P_sep (nappes courbes si > 1)
    "n_terms_poly": 20,
    "domain":       (0.0, 10.0),
    "epsilon":      0.5,        # écart des niveaux (en unités de std(P) = 1)
    "proj_iter":    50,
    "proj_tol":     1e-10,

    # ── DONNÉES ───────────────────────────────────────────────────────────────
    "n_train":      10,
    "n_test":       2000,
    "n_unlabeled":  3000,
    "seeds": {42: 48154, 43: 2252, 44: 115561, 99: 9435199},

    # ── NORME DE SOBOLEV (classification) ─────────────────────────────────────
    "weights":      {0: 1e-4, 1: 1, 2: 0.5, 3: 0.15},   # espace ambiant
    "weights_emb":  {0: 1e-4, 1: 1, 2: 0, 3: 0},  # même norme que l'ambiant

    # ── CLASSIFICATION QP (semi-interpolation) ────────────────────────────────
    "qp_margin":   1.0,
    "deg_clf_avant": 2,        # degré du classifieur dans l'espace ambiant
    "deg_clf_apres": 2,        # degré dans l'espace plongé (bas : test du
                               #  redressement — si le plongement marche, ça suffit)
    "thres1":      1e-25,      # bande spectrale base H-orthonormée
    "thres2":      1e25,
    "const_pen":   0,       # ancrage de la constante libre
    "n_G":         1500,       # points cloud pour la Gram du classifieur

    # ── PLONGEMENT (programme historique) ─────────────────────────────────────
    "n_layers":    2,
    "embed_dim":   10,         # vecteurs propres retenus par couche
    "lambda_G":    1.0,        # régularisation de la Gram du plongement
    "k_proj":      12,          # dim des projections (plafonné à la dim d'entrée ;
                               #   n_proj = ceil(d/k_proj), DÉRIVÉ : les blocs
                               #   orthogonaux partitionnent l'espace -> le span
                               #   contient tous les polynômes de degré 1)
    "emb_thres1":  0.9,       # bande spectrale du plongement (relative à s_max) :
    "emb_thres2": 55,        #   on garde emb_thres1 < s/s_max < emb_thres2
                               #   (plafond : les grandes VP sont à jeter)
    "n_deg":       5,          # degré des polynômes du plongement
    "activation":  "none",
    "weights_plg": {0: 0, 1: 1, 2: 0, 3: 0},   # poids Sobolev du plongement
}

d   = params["n_ambiant"]
eps = params["epsilon"]
DOM_LO, DOM_HI = params["domain"]

# ══════════════════════════════════════════════════════════════════════════════
# POLYNÔMES
# ══════════════════════════════════════════════════════════════════════════════
def random_sparse_polynomial(dd, degree, n_terms, seed=None):
    rng = np.random.default_rng(seed)
    all_indices = [e for e in product(range(degree+1), repeat=dd) if sum(e) <= degree]
    n_terms = min(n_terms, len(all_indices))
    indices = rng.choice(all_indices, size=n_terms, replace=False)
    coeffs  = rng.normal(size=n_terms)
    def P(x):
        y = np.zeros(x.shape[0])
        for c, alpha in zip(coeffs, indices):
            t = np.ones(x.shape[0])
            for j, e in enumerate(alpha):
                if e > 0: t *= x[:, j]**e
            y += c*t
        return y
    return P, indices, coeffs

P_raw, idx_sep, coeffs_sep = random_sparse_polynomial(
    d, params["deg_P"], params["n_terms_poly"], seed=params["seeds"][99])

# ── RENORMALISATION SUR LE DOMAINE : std(P)=1, centré ─────────────────────────
# (normaliser les coefficients ne suffit pas : à deg_P>=3, les termes de haut
#  degré font exploser P sur le cube et ε=0.7 devient invisible.)
_Xs = np.random.default_rng(12345).uniform(DOM_LO, DOM_HI, (200000, d))
_v = P_raw(_Xs); _scale = float(_v.std()); _shift = float(_v.mean())
def P_sep(X): return (P_raw(X) - _shift) / _scale
print(f"P_sep renormalisé : std=1 sur le domaine (facteur {_scale:.3g}, centre {_shift:.3g})")

# ══════════════════════════════════════════════════════════════════════════════
# CLOUD SUR LES DEUX NAPPES  (Gauss-Newton, 50/50, AVEC labels)
# ══════════════════════════════════════════════════════════════════════════════
def grad_P(X, h=1e-6):
    G = np.empty_like(X)
    for k in range(d):
        Xp = X.copy(); Xp[:, k] += h
        Xm = X.copy(); Xm[:, k] -= h
        G[:, k] = (P_sep(Xp) - P_sep(Xm)) / (2*h)
    return G

def project_to_level(X, t):
    Xp = X.copy()
    for _ in range(params["proj_iter"]):
        r = P_sep(Xp) - t
        if np.max(np.abs(r)) < params["proj_tol"]: break
        G = grad_P(Xp); g2 = np.sum(G**2, axis=1)
        g2 = np.where(g2 > 1e-14, g2, 1e-14)
        Xp = Xp - (r/g2)[:, None] * G
    return Xp

def sample_on_level(t, n_target, seed, max_tries=2000000):
    rng = np.random.default_rng(seed); out = []; got = 0; tries = 0
    while got < n_target and tries < max_tries:
        m = max(8*(n_target-got), 1000)
        X = rng.uniform(DOM_LO, DOM_HI, (m, d))
        Xp = project_to_level(X, t)
        ok = np.all((Xp >= DOM_LO) & (Xp <= DOM_HI), axis=1) \
           & (np.abs(P_sep(Xp) - t) < params["proj_tol"])
        g = Xp[ok]
        if len(g): out.append(g); got += len(g)
        tries += m
    if got < n_target:
        raise RuntimeError(f"nappe t={t}: {got}/{n_target} points")
    return np.vstack(out)[:n_target]

def sample_two_levels(n_target, seed):
    """points 50/50 sur les deux nappes + labels ±1, mélangés."""
    n0 = n_target // 2; n1 = n_target - n0
    X0 = sample_on_level(0.0, n0, seed)
    X1 = sample_on_level(eps, n1, seed+1)
    X = np.vstack([X0, X1])
    y = np.concatenate([-np.ones(n0), np.ones(n1)])
    perm = np.random.default_rng(seed+2).permutation(len(X))
    return X[perm], y[perm]

print(f"Génération du cloud {{P=0}} ∪ {{P=ε}}  (deg_P={params['deg_P']}, ε={eps})...")
X_train, y_train = sample_two_levels(params["n_train"], params["seeds"][42])
X_test,  y_test  = sample_two_levels(params["n_test"],  params["seeds"][43])
X_unlab, y_unlab = sample_two_levels(params["n_unlabeled"], params["seeds"][44])
X_all = np.vstack([X_train, X_unlab]); N_all = len(X_all)
y_all = np.concatenate([y_train, y_unlab])   # labels des nappes du cloud (diagnostic
                                             # uniquement — jamais utilisés pour fitter)
for nom, Xc, yc in [("train", X_train, y_train), ("test", X_test, y_test)]:
    p = P_sep(Xc)
    resid = np.minimum(np.abs(p), np.abs(p-eps)).max()
    print(f"  {nom:5s}: {len(Xc)} pts  nappe0={int((yc<0).sum())} nappeε={int((yc>0).sum())}"
          f"  max|résidu|={resid:.2e}")

# ══════════════════════════════════════════════════════════════════════════════
# DIAGNOSTIC DE SÉPARATION  (avant / après plongement)
#   ratio inter/intra : distance moyenne entre nappes / distance moyenne au sein
#   d'une nappe. > 1 = les nappes s'éloignent. On mesure sur le cloud (labels de
#   génération, diagnostic pur).
# ══════════════════════════════════════════════════════════════════════════════
def separation_diag(Z, y, n_sub=800, seed=0, titre=""):
    rng = np.random.default_rng(seed)
    i0 = np.where(y < 0)[0]; i1 = np.where(y > 0)[0]
    i0 = rng.choice(i0, min(n_sub, len(i0)), replace=False)
    i1 = rng.choice(i1, min(n_sub, len(i1)), replace=False)
    Z0, Z1 = Z[i0], Z[i1]
    def mdist(A, B):
        return np.mean(np.sqrt(np.maximum(
            np.sum(A**2,1)[:,None] + np.sum(B**2,1)[None,:] - 2*A@B.T, 0)))
    intra = 0.5*(mdist(Z0, Z0) + mdist(Z1, Z1))
    inter = mdist(Z0, Z1)
    # 1-NN inter-nappe : fraction de points dont le plus proche voisin est de
    # l'AUTRE nappe (0 = nappes bien séparées, ~0.5 = mélangées)
    Zs = np.vstack([Z0, Z1]); ys = np.concatenate([-np.ones(len(Z0)), np.ones(len(Z1))])
    D = np.sum(Zs**2,1)[:,None] + np.sum(Zs**2,1)[None,:] - 2*Zs@Zs.T
    np.fill_diagonal(D, np.inf)
    nn = np.argmin(D, axis=1)
    frac_cross = np.mean(ys[nn] != ys)
    print(f"  [{titre}] inter/intra = {inter/intra:.4f}   "
          f"(inter={inter:.3f} intra={intra:.3f})   1-NN croisé = {frac_cross:.3f}")
    return inter/intra, frac_cross

# ══════════════════════════════════════════════════════════════════════════════
# CLASSIFICATEUR QP  (semi-interpolation Sobolev, expert polynomial générique)
#   min ‖u‖²_H  s.c.  y_i·u(x_i) >= marge,  base H-orthonormée + constante libre.
#   Générique : X de dimension quelconque (ambiant OU plongé).
# ══════════════════════════════════════════════════════════════════════════════
def solve_qp(A, y, G, margin, verbose=False):
    n, k = A.shape
    Gr = G + 1e-12*np.eye(k)
    con = LinearConstraint(np.diag(y)@A, lb=margin, ub=np.inf)
    try:    c0 = np.linalg.lstsq(A, 1.5*margin*y, rcond=None)[0]
    except Exception: c0 = np.zeros(k)
    res = minimize(lambda c: c@Gr@c, c0, jac=lambda c: 2*Gr@c,
                   constraints=[con], method='SLSQP',
                   options={'maxiter': 500, 'ftol': 1e-11})
    c = res.x
    marge_eff = float(np.min(y*(A@c)))
    return c, marge_eff >= margin - 1e-4, marge_eff

def classify_poly_qp(X_tr, y_tr, X_te, y_te, X_cloud, deg, weights, titre=""):
    """Classifieur polynomial par semi-interpolation QP. Retourne (acc, auc)."""
    n, dd = X_tr.shape
    # normalisation par coordonnée -> [-1,1] (calculée sur le cloud)
    lo = X_cloud.min(axis=0); hi = X_cloud.max(axis=0)
    span = np.where(hi - lo > 1e-12, hi - lo, 1.0)
    def to_u(X): return 2*(X - lo)/span - 1.0
    SC = 2.0/span                                      # ∂u/∂x par coordonnée

    poly = PolynomialFeatures(degree=deg, include_bias=False)
    Phi_tr = poly.fit_transform(to_u(X_tr))
    powers = poly.powers_; n_feat = Phi_tr.shape[1]

    def pderiv(U, coefs, dims=()):
        P = powers.astype(float).copy(); m = coefs.astype(float).copy()
        for dm in dims: m = m*P[:, dm]; P[:, dm] -= 1
        v = m != 0
        return np.zeros(len(U)) if not v.any() else (U[:, None, :]**P[None, v, :]).prod(2)@m[v]

    rng = np.random.default_rng(4242)
    idx = rng.choice(len(X_cloud), min(params["n_G"], len(X_cloud)), replace=False)
    U_G = to_u(X_cloud[idx]); nG = len(idx)
    PHI = poly.transform(U_G)
    E = np.eye(n_feat)
    w0 = weights.get(0, 0.); w1 = weights.get(1, 0.); w2 = weights.get(2, 0.)
    G = np.zeros((n_feat, n_feat))
    if w0: G += w0*(PHI.T@PHI)/nG
    if w1:
        GG = np.zeros((nG, n_feat, dd))
        for j in range(n_feat):
            for a in range(dd):
                if powers[j, a] > 0:
                    GG[:, j, a] = SC[a]*pderiv(U_G, E[j], (a,))
        G += w1*np.einsum('xik,xjk->ij', GG, GG)/nG
    if w2:
        HH = np.zeros((nG, n_feat, dd, dd))
        for j in range(n_feat):
            for a in range(dd):
                for b in range(a, dd):
                    if (powers[j, a] > 0 and powers[j, b] > 0) or (a == b and powers[j, a] > 1):
                        v = SC[a]*SC[b]*pderiv(U_G, E[j], (a, b))
                        HH[:, j, a, b] = v; HH[:, j, b, a] = v
        G += w2*np.einsum('xikl,xjkl->ij', HH, HH)/nG

    # base H-orthonormée (bande) + centrage + constante libre ancrée
    mean_phi = PHI.mean(axis=0)
    s_g, V_g = np.linalg.eigh(G); smax = 1 #s_g.max()
    keep = (s_g > smax*params["thres1"]) & (s_g < smax*params["thres2"])
    if keep.sum() == 0:
        raise ValueError("bande spectrale vide")
    T = V_g[:, keep]/np.sqrt(s_g[keep]); r = int(keep.sum())
    A_qp = np.hstack([(Phi_tr - mean_phi)@T, np.ones((n, 1))])
    G_qp = np.eye(r+1); G_qp[-1, -1] = params["const_pen"]
    sol, feas, marge = solve_qp(A_qp, y_tr, G_qp, params["qp_margin"])
    coef = T@sol[:r]; off = sol[-1] - mean_phi@coef

    f_te = poly.transform(to_u(X_te))@coef + off
    f_tr = Phi_tr@coef + off
    def acc(f, y):
        sg = np.sign(f)
        return np.mean(np.where(sg == 0, 0.5, (sg == np.sign(y)).astype(float)))
    try:    auc = roc_auc_score((y_te > 0).astype(int), f_te)
    except Exception: auc = float('nan')
    print(f"  [{titre}] deg={deg} feat={n_feat} rang={r} | faisable={feas} "
          f"marge={marge:.3f} ‖u‖_H={np.sqrt(max(sol[:r]@sol[:r],0)):.4f} | "
          f"acc_tr={acc(f_tr,y_tr):.2f} acc_te={acc(f_te,y_te):.4f} AUC={auc:.4f}")
    return acc(f_te, y_te), auc

def classify_ridge(X_tr, y_tr, X_te, y_te, deg, titre=""):
    poly = PolynomialFeatures(degree=deg)
    r = Ridge(alpha=1e-8).fit(poly.fit_transform(X_tr), y_tr)
    f = r.predict(poly.transform(X_te))
    a = np.mean(np.sign(f) == np.sign(y_te))
    try:    auc = roc_auc_score((y_te > 0).astype(int), f)
    except Exception: auc = float('nan')
    print(f"  [{titre}] Ridge deg={deg} : acc_te={a:.4f} AUC={auc:.4f}")
    return a, auc

# ══════════════════════════════════════════════════════════════════════════════
# PLONGEMENT SPECTRAL  (programme historique, repris tel quel)
# ══════════════════════════════════════════════════════════════════════════════
def activate(X, mode):
    if mode == "tanh":    return np.tanh(X)
    if mode == "sigmoid": return 1/(1 + np.exp(-X))
    return X

def make_projections_ortho(dd, k_proj, rng):
    """Projections sur des sous-espaces ORTHOGONAUX deux à deux qui PARTITIONNENT
    R^dd. Construction : QR d'une matrice gaussienne dd×dd -> base orthonormée
    aléatoire ; on la découpe en blocs de k_proj lignes. Le dernier bloc a
    dimension dd mod k_proj s'il ne divise pas (choix unique : l'orthogonal des
    précédents). k_proj est PLAFONNÉ à dd (au-delà, les vecteurs propres seraient
    redondants pour beaucoup de calcul en plus).
    GARANTIE : la concaténation des bases relevées contient TOUS les polynômes de
    degré 1 dans son span — les lignes des blocs forment une base de R^dd, donc
    toute forme linéaire x ↦ v·x est combinaison des coordonnées projetées."""
    k = min(k_proj, dd)
    M = rng.standard_normal((dd, dd))
    Q, _ = np.linalg.qr(M)                     # lignes de Q.T = base orthonormée
    B = Q.T                                    # (dd, dd) : chaque ligne unitaire
    blocks = [B[i:i+k] for i in range(0, dd, k)]
    return blocks                              # liste de (k_p, dd), k_p<=k

def poly_features_projected(X, Pis, n_deg):
    """Pis : liste de matrices (k_p, d). Monômes de degré <= n_deg par projection
    (k_p peut varier -> monômes par bloc). Retourne Phi concaténée + la liste des
    monômes par bloc."""
    n = X.shape[0]
    cols = []; monomes_list = []
    for Pi_p in Pis:
        k_p = Pi_p.shape[0]
        Zp = X @ Pi_p.T                        # (n, k_p)
        monomes = [a for a in product(range(n_deg+1), repeat=k_p) if 0 < sum(a) <= n_deg]
        monomes_list.append(monomes)
        Phi_p = np.ones((n, len(monomes)))
        for m, alpha in enumerate(monomes):
            for j, e in enumerate(alpha):
                if e > 0: Phi_p[:, m] *= Zp[:, j]**e
        cols.append(Phi_p)
    return np.hstack(cols), monomes_list

def poly_gram_projected(X_cloud, Pis, n_deg, weights, lambda_G):
    n = X_cloud.shape[0]
    sizes = []
    monomes_list = []
    for Pi_p in Pis:
        k_p = Pi_p.shape[0]
        monomes = [a for a in product(range(n_deg+1), repeat=k_p) if 0 < sum(a) <= n_deg]
        monomes_list.append(monomes); sizes.append(len(monomes))
    n_funcs = sum(sizes)
    G = np.zeros((n_funcs, n_funcs))
    off = 0
    for Pi_p, monomes in zip(Pis, monomes_list):
        k_p = Pi_p.shape[0]; n_mon = len(monomes)
        sl = slice(off, off + n_mon); off += n_mon
        Zp = X_cloud @ Pi_p.T
        Phi_p = np.ones((n, n_mon))
        for m, alpha in enumerate(monomes):
            for j, e in enumerate(alpha):
                if e > 0: Phi_p[:, m] *= Zp[:, j]**e
        w0 = weights.get(0, 0.)
        if w0: G[sl, sl] += w0*(Phi_p.T@Phi_p)/n
        w1 = weights.get(1, 0.)
        if w1:
            GradZ = np.zeros((n, n_mon, k_p))
            for m, alpha in enumerate(monomes):
                for j in range(k_p):
                    if alpha[j] > 0:
                        col = np.ones(n)*alpha[j]
                        for l, e in enumerate(alpha):
                            e2 = e - 1 if l == j else e
                            if e2 > 0: col *= Zp[:, l]**e2
                        GradZ[:, m, j] = col
            GradX = np.einsum('nmj,jd->nmd', GradZ, Pi_p)
            G[sl, sl] += w1*np.einsum('nmi,nli->ml', GradX, GradX)/n
    G += lambda_G*np.eye(n_funcs)
    return G

def poly_embedding(X_input, X_cloud, weights, lambda_G,
                   k_proj, n_deg, embed_dim, rng,
                   emb_thres1, emb_thres2):
    """Plongement spectral. Projections orthogonales partitionnant l'espace
    (n_proj dérivé = ceil(d/k_proj)). Sélection : on garde les embed_dim DERNIERS
    vecteurs propres disponibles SOUS le seuil plafond emb_thres2 (les plus
    petites VP sous le plafond — celles qui varient le moins sur le cloud).
    emb_thres1 n'est plus utilisé (pas de plancher).
    La dimension d'image est min(embed_dim, VP disponibles sous plafond)."""
    Pis = make_projections_ortho(X_input.shape[1], k_proj, rng)
    print(f"  projections orthogonales : {len(Pis)} blocs de dims "
          f"{[p.shape[0] for p in Pis]} (partition de R^{X_input.shape[1]})")
    Phi_cloud, monomes = poly_features_projected(X_cloud, Pis, n_deg)
    G = poly_gram_projected(X_cloud, Pis, n_deg, weights, lambda_G)
    eigvals, eigvecs = np.linalg.eigh(G)
    order = np.argsort(eigvals)[::-1]
    eigvals = eigvals[order]; eigvecs = eigvecs[:, order]
    smax = 1  # échelle absolue (comme avant)
    # sélection : SOUS le plafond, on prend les embed_dim DERNIERS (plus petits)
    under_ceiling = np.where(eigvals < emb_thres2*smax)[0]
    if len(under_ceiling) == 0:
        raise ValueError(f"aucune VP sous le plafond {emb_thres2} ; ajuster emb_thres2")
    # under_ceiling est trié par VP décroissante ; les DERNIERS = les plus petits
    idx_sel = under_ceiling[-embed_dim:] if len(under_ceiling) > embed_dim else under_ceiling
    V = eigvecs[:, idx_sel]; L = eigvals[idx_sel]
    n_above = int((eigvals >= emb_thres2*smax).sum())
    print(f"  spectre : {len(eigvals)} VP | jetées en HAUT (s>={emb_thres2:g}) : {n_above} | "
          f"disponibles sous plafond : {len(under_ceiling)} | retenues : {len(L)}")
    print(f"  VP gardées ({len(L)}) : {np.array2string(L, precision=3, max_line_width=100)}")
    V_norm = V/np.sqrt(L)[None, :]
    embed_cld = Phi_cloud@V_norm
    embed_means = embed_cld.mean(axis=0)
    Phi_in, _ = poly_features_projected(X_input, Pis, n_deg)
    return Phi_in@V_norm - embed_means, Pis, V_norm, embed_means

# ══════════════════════════════════════════════════════════════════════════════
# EXPÉRIENCE
# ══════════════════════════════════════════════════════════════════════════════
# centrage (comme le programme historique) — le plongement travaille centré
center = X_all.mean(axis=0)
Xc_train = X_train - center; Xc_test = X_test - center; Xc_all = X_all - center

print("\n" + "="*72)
print("AVANT PLONGEMENT (espace ambiant)")
print("="*72)
sep0 = separation_diag(Xc_all, y_all, titre="séparation ambiant")
acc0, auc0 = classify_poly_qp(Xc_train, y_train, Xc_test, y_test, Xc_all,
                              params["deg_clf_avant"], params["weights"],
                              titre="QP ambiant")
classify_ridge(Xc_train, y_train, Xc_test, y_test, 3, titre="baseline ambiant")

print("\n" + "="*72)
print(f"PLONGEMENT SPECTRAL ({params['n_layers']} couche(s), "
      f"embed_dim={params['embed_dim']}, k_proj={params['k_proj']} (blocs orthogonaux), "
      f"n_deg={params['n_deg']}, bande=[{params['emb_thres1']:g},{params['emb_thres2']:g}])")
print("="*72)
rng = np.random.default_rng(0)
E_train, E_test, E_all = Xc_train, Xc_test, Xc_all
for layer in range(params["n_layers"]):
    print(f"── couche {layer+1} ──")
    E_all_new, Pis, V_norm, means = poly_embedding(
        E_all, E_all, params["weights_plg"], params["lambda_G"],
        params["k_proj"], params["n_deg"],
        params["embed_dim"], rng,
        params["emb_thres1"], params["emb_thres2"])
    def embed_new(Xin, Pis=Pis, V=V_norm, m=means, nd=params["n_deg"]):
        Phi, _ = poly_features_projected(Xin, Pis, nd)
        return activate(Phi@V - m, params["activation"])
    E_train = embed_new(E_train); E_test = embed_new(E_test)
    E_all = activate(E_all_new, params["activation"])
    print(f"  dims : {E_all.shape[1]}")
    # stats à chaque couche (utile en multi-layers)
    separation_diag(E_all, y_all, titre=f"séparation couche {layer+1}")
    classify_ridge(E_train, y_train, E_test, y_test, 2, titre=f"Ridge couche {layer+1}")

print("\n" + "="*72)
print("APRÈS PLONGEMENT (espace plongé)")
print("="*72)
sep1 = separation_diag(E_all, y_all, titre="séparation plongé")
acc1, auc1 = classify_poly_qp(E_train, y_train, E_test, y_test, E_all,
                              params["deg_clf_apres"], params["weights_emb"],
                              titre="QP plongé")
classify_ridge(E_train, y_train, E_test, y_test, 2, titre="baseline plongé")

print("\n" + "="*72)
print("BILAN")
print("="*72)
print(f"  séparation inter/intra : {sep0[0]:.4f} -> {sep1[0]:.4f}   "
      f"({'ÉLOIGNEMENT' if sep1[0] > sep0[0] else 'pas d’éloignement'})")
print(f"  1-NN croisé            : {sep0[1]:.3f} -> {sep1[1]:.3f}   "
      f"(plus petit = mieux séparé)")
print(f"  QP  acc_te             : {acc0:.4f} (deg {params['deg_clf_avant']}, ambiant) "
      f"-> {acc1:.4f} (deg {params['deg_clf_apres']}, plongé)")


P_sep renormalisé : std=1 sur le domaine (facteur 5.16e+04, centre 2.29e+04)
Génération du cloud {P=0} ∪ {P=ε}  (deg_P=5, ε=0.5)...
  train: 10 pts  nappe0=5 nappeε=5  max|résidu|=2.22e-16
  test : 2000 pts  nappe0=1000 nappeε=1000  max|résidu|=1.44e-15

AVANT PLONGEMENT (espace ambiant)
  [séparation ambiant] inter/intra = 1.0144   (inter=6.492 intra=6.400)   1-NN croisé = 0.048
  [QP ambiant] deg=2 feat=14 rang=14 | faisable=True marge=1.000 ‖u‖_H=1.2726 | acc_tr=1.00 acc_te=0.6890 AUC=0.6765
  [baseline ambiant] Ridge deg=3 : acc_te=0.5520 AUC=0.5870

PLONGEMENT SPECTRAL (2 couche(s), embed_dim=10, k_proj=12 (blocs orthogonaux), n_deg=5, bande=[0.9,55])
── couche 1 ──
  projections orthogonales : 1 blocs de dims [4] (partition de R^4)
  spectre : 125 VP | jetées en HAUT (s>=55) : 80 | disponibles sous plafond : 45 | retenues : 10
  VP gardées (10) : [1.185 1.181 1.162 1.117 1.092 1.081 1.078 1.006 1.002 1.001]
  dims : 10
  [séparation couche 1] inter/intra = 1.0575   (inter=1.830 i